## Project -> Question & Answering on Private Documents

In [1]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)

True

### 1. Load Documents from a file and online

In [2]:
def load_documents(file):
    name, extension = os.path.splitext(file)
    if extension == ".pdf":
        print(f"Loading PDF document: {name}")
        from langchain_community.document_loaders import PyPDFLoader
        loader = PyPDFLoader(file)
    elif extension == ".docx":
        print(f"Loading Word document: {name}")
        from langchain_community.document_loaders import Docx2txtLoader
        loader = Docx2txtLoader(file)
    else:
        print("Unable to load the file")
        return None
    data = loader.load()
    return data

In [3]:
from langchain_community.document_loaders import WikipediaLoader
def load_from_wikipedia(query, max_docs=2, lang='en'):
    print(f"Loading document from WIkipedia for {query}")
    loader = WikipediaLoader(query=query, 
                           load_max_docs=max_docs,
                           lang=lang
                          )
    data = loader.load()
    return data

In [9]:
data = load_documents('files/us_constitution.pdf')

Loading PDF document: files/us_constitution


In [10]:
print(data[1].page_content)

TheHouseofRepresentativesshallbecomposedofMemberschosen
everysecondYearbythePeopleoftheseveralStates,andthe
ElectorsineachStateshallhavetheQualificationsrequisitefor
ElectorsofthemostnumerousBranchoftheStateLegislature.
NoPersonshallbeaRepresentativewhoshallnothaveattainedtothe
AgeoftwentyfiveYears,andbeensevenYearsaCitizenoftheUnited
States,andwhoshallnot,whenelected,beanInhabitantofthatState
inwhichheshallbechosen.
RepresentativesanddirectTaxesshallbeapportionedamongthe
severalStateswhichmaybeincludedwithinthisUnion,accordingto
theirrespectiveNumbers,whichshallbedeterminedbyaddingtothe
wholeNumberoffreePersons,includingthoseboundtoServicefora
TermofYears,andexcludingIndiansnottaxed,threefifthsofallother
Persons.TheactualEnumerationshallbemadewithinthreeYears
afterthefirstMeetingoftheCongressoftheUnitedStates,andwithin
everysubsequentTermoftenYears,insuchMannerastheyshallby
Lawdirect.ThenumberofRepresentativesshallnotexceedonefor
everythirtyThousand,buteachStateshallhaveatLeastone
Rep

In [11]:
data = load_from_wikipedia("GPT-4")

Loading document from WIkipedia for GPT-4


In [12]:
print(data[1].page_content)

GPT-4.5 (codenamed "Orion") is a large language model developed by OpenAI as part of the GPT series. Officially released on February 27, 2025, GPT-4.5 is available to users subscribed to the ChatGPT Plus and Pro plans across web, mobile, and desktop platforms. Access was also provided through the OpenAI API and Developer Playground until July 14, 2025. On August 7, 2025, with the release of GPT-5, GPT-4.5 was removed from both the API and ChatGPT users with the Plus and Teams tier, while Pro users are still able to access to GPT-4.5 model under the "Legacy Models" tab.


== Overview ==
GPT-4.5 was primarily trained using unsupervised learning, which improves its ability to recognize patterns, draw connections, and generate creative insights without reasoning. This method was combined with supervised fine-tuning and reinforcement learning from human feedback. The computational resources needed for training were provided by Microsoft Azure.
Sam Altman described GPT-4.5 as a "giant, expen

### 2. Chunking the loaded documents

In [4]:
data = load_documents("files/us_constitution.pdf")
from langchain_text_splitters import RecursiveCharacterTextSplitter
def chunk_data(data, chunk_size=256):
    print(f"Chunking the data into chunk size {chunk_size}")
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size)
    chunks = splitter.split_documents(data)

    return chunks

chunks = chunk_data(data)

Loading PDF document: files/us_constitution
Chunking the data into chunk size 256


In [5]:
print(len(chunks))

513


In [6]:
# Calculate the cost of embeddings of all the chunks
def price_embedding_cost(texts):
    import tiktoken
    enc = tiktoken.encoding_for_model("text-embedding-ada-002")
    total_tokens = sum([len(enc.encode(page.page_content)) for page in texts])
    print(f"Total tokens: {total_tokens}")
    print(f"Cost: ${total_tokens / 1000 * 0.0004:.6f}")
price_embedding_cost(chunks)

Total tokens: 34048
Cost: $0.013619


### 3. Embeddings and uploading to a Vector Database (Pinecone)

In [7]:
def insert_or_fetch_embeddings(index_name, chunks):
    import pinecone # Used to connect to pinecone directly
    from langchain_community.vectorstores import Pinecone # DEPRICATED
    from langchain_pinecone import PineconeVectorStore
    from langchain_openai import OpenAIEmbeddings
    from pinecone import ServerlessSpec

    pc = pinecone.Pinecone()
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=1536)
    # If Index already exists in Pinecone just retireve the vector store using langchain -> Pinecone
    if index_name in pc.list_indexes().names():
        print(f"Index {index_name} already exists. Loading embeddings ...", end='')
        vector_store = PineconeVectorStore.from_existing_index(index_name, embeddings)
        print("OK")
        return vector_store
    else:
        print(f"Creatng index {index_name} and embeddings ...", end='')
        pc.create_index(
            name=index_name,
            dimension=1536,
            metric="cosine",
            spec=ServerlessSpec(cloud='aws', region='us-east-1')
        )
        vector_store = PineconeVectorStore.from_documents(chunks, embeddings, index_name=index_name)
        print("OK")
        return vector_store

In [8]:
def delete_pinecone_index(index_name="all"):
    import pinecone
    pc = pinecone.Pinecone()
    if index_name == "all":
        indexes = pc.list_indexes().names()
        print("Deleting all indexes ...")
        for index in indexes:
            pc.delete_index(index)
        print("OK")
    else:
        print(f"Deleting index {index_name} ...", end='')
        pc.delete_index(index_name)
        print("OK")

In [9]:
# Cleaning pinecone before adding more indexes
delete_pinecone_index()

Deleting all indexes ...
OK


In [10]:
index_name = "askadocument"
vector_store = insert_or_fetch_embeddings(index_name, chunks)

Creatng index askadocument and embeddings ...OK


### 4. Asking a Question and getting an answer
1. Create a retriever using the vector store to get the relevant vectors.
2. Initialise the LLM
3. Create a chain with the LLM and the retiever with chain_type as stuff
4. Invoke the chain with the question to get the answer. (This will send all the information to the LLM and provide the most relevent answer depending on the context it is being provided.

In [11]:
def ask_and_get_answer(vector_store, q):
    from langchain_classic.chains import RetrievalQA
    from langchain_openai import ChatOpenAI
    # 1. Retriever
    retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={'k': 5})
    # 2. LLM
    llm = ChatOpenAI(model="gpt-4o", temperature=1)
    # 3. Chain
    chain = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=retriever)
    # 4. Invoke
    answer = chain.invoke(q)

    return answer
    

In [12]:
q = "What is the document about?"
answer = ask_and_get_answer(vector_store, q)
print(answer)

{'query': 'What is the document about?', 'result': 'The document is a section of the United States Constitution. It specifically refers to Article IV, which deals with the relations between states and the federal government. It includes provisions such as the "Full Faith and Credit Clause," requiring states to recognize the public acts, records, and judicial proceedings of other states, and the "Privileges and Immunities Clause," ensuring that citizens of each state are entitled to the same privileges as citizens in other states. Additionally, it touches on Congress\'s authority to manage federal property and territories.'}


### 5. Using ChromaDB

In [13]:
def create_embeddings_chroma(chunks, persist_directory="./chroma_db"):
    from langchain_chroma import Chroma
    from langchain_openai import OpenAIEmbeddings

    embeddings = OpenAIEmbeddings(model='text-embedding-3-small', dimensions=1536)
    vector_store = Chroma.from_documents(chunks, embeddings, persist_directory=persist_directory)
    return vector_store

def load_embeddings_chroma(persist_directory="./chroma_db"):
    from langchain_chroma import Chroma
    from langchain_openai import OpenAIEmbeddings

    embeddings = OpenAIEmbeddings(model='text-embedding-3-small', dimensions=1536)
    vector_store = Chroma(persist_directory=persist_directory, embedding_function=embeddings)
    return vector_store
    

In [14]:
data = load_documents('files/rag_powered_by_google_search.pdf')
chunks = chunk_data(data, chunk_size=256)
vector_store = create_embeddings_chroma(chunks)

Loading PDF document: files/rag_powered_by_google_search
Chunking the data into chunk size 256


In [15]:
q = 'What is Vertex AI search?'
answer = ask_and_get_answer(vector_store, q)
print(answer['result'])

Vertex AI Search is a search platform that offers features such as customizable answers, search tuning, vector search, grounding, and compliance updates for enterprises. It uses Tensor Processing Units (TPUs) to power its large-scale semantic search capabilities.


In [16]:
db = load_embeddings_chroma()
q = "How many pairs of questions and answers had the StackOverflow dataset?"
answer = ask_and_get_answer(vector_store, q)
print(answer['result'])

The StackOverflow dataset had 8 million pairs of questions and answers.


In [17]:
q = "Multiply that number by 2"
answer = ask_and_get_answer(vector_store, q)
print(answer['result'])
# Make a note there is no persistent memory assigned yet so it couldn't find the number to double.
# The Follow up question did not work.

I'm sorry, I am not aware of the specific number you are referring to. Could you please provide more context or specify the number you would like doubled?


### 6. Adding Memory (Chat History)
The first 2 steps are the same.  
3. Initialise a memory object  
4. Use a chain which can handle memory and pass the memory when initialising the chain.  
5. Define ask_question function to invoke the chain.     
The function is defined seperately because we dont want to initialise and reset the chain again when we ask a new question.  

In [20]:
# Load the data 
data = load_documents('files/rag_powered_by_google_search.pdf')
chunks = chunk_data(data, chunk_size=256)
vector_store = create_embeddings_chroma(chunks)

Loading PDF document: files/rag_powered_by_google_search
Chunking the data into chunk size 256


In [18]:
from langchain_openai import ChatOpenAI
from langchain_classic.chains import ConversationalRetrievalChain  # Import class for building conversational AI chains 
from langchain_classic.memory import ConversationBufferMemory  # Import memory for storing conversation history

# Instantiate a ChatGPT LLM (temperature controls randomness)
llm = ChatOpenAI(model_name='gpt-3.5-turbo', temperature=0)  

# Configure vector store to act as a retriever (finding similar items, returning top 5)
retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={'k': 5})  


# Create a memory buffer to track the conversation
memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)

crc = ConversationalRetrievalChain.from_llm(
    llm=llm,  # Link the ChatGPT LLM
    retriever=retriever,  # Link the vector store based retriever
    memory=memory,  # Link the conversation memory
    chain_type='stuff',  # Specify the chain type
    verbose=False  # Set to True to enable verbose logging for debugging
)


C:\Users\ashut\AppData\Local\Temp\ipykernel_27192\1230738600.py:13: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)


In [19]:
def ask_question(q, chain):
    result = chain.invoke({'question': q})
    return result

In [21]:
q = 'How many pairs of questions and answers has the StackOverflow dataset?'
result = ask_question(q, crc)
print(result['answer'])

The StackOverflow dataset mentioned in the context contains 8 million pairs of questions and answers.


In [22]:
q = "Multiply the number by 10."
result = ask_question(q, crc)
print(result['answer'])

The result of multiplying the number of pairs of questions and answers in the StackOverflow dataset by 10 would be 80 million pairs of questions and answers.


In [23]:
q = "Divide the result by 80"
result = ask_question(q, crc)
print(result['answer'])

The result of dividing 80 million pairs of questions and answers by 80 is 1 million pairs of questions and answers.


In [24]:
for item in result['chat_history']:
    print(item)

content='How many pairs of questions and answers has the StackOverflow dataset?' additional_kwargs={} response_metadata={}
content='The StackOverflow dataset mentioned in the context contains 8 million pairs of questions and answers.' additional_kwargs={} response_metadata={} tool_calls=[] invalid_tool_calls=[]
content='Multiply the number by 10.' additional_kwargs={} response_metadata={}
content='The result of multiplying the number of pairs of questions and answers in the StackOverflow dataset by 10 would be 80 million pairs of questions and answers.' additional_kwargs={} response_metadata={} tool_calls=[] invalid_tool_calls=[]
content='Divide the result by 80' additional_kwargs={} response_metadata={}
content='The result of dividing 80 million pairs of questions and answers by 80 is 1 million pairs of questions and answers.' additional_kwargs={} response_metadata={} tool_calls=[] invalid_tool_calls=[]


### 7. Using a Custom Prompt

In [33]:
from langchain_openai import ChatOpenAI
from langchain_classic.chains import ConversationalRetrievalChain  # Import class for building conversational AI chains 
from langchain_classic.memory import ConversationBufferMemory  # Import memory for storing conversation history

from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
# Instantiate a ChatGPT LLM (temperature controls randomness)
llm = ChatOpenAI(model_name='gpt-3.5-turbo', temperature=0)  

# Configure vector store to act as a retriever (finding similar items, returning top 5)
retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={'k': 5})  


# Create a memory buffer to track the conversation
memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)

system_template = r'''
Use the following pieces of context to answer the user's question.
---------------------
Context: ```{context}```
'''

user_template = '''
Question: ```{question}```
Chat History: ```{chat_history}```
'''

messages = [
    SystemMessagePromptTemplate.from_template(system_template),
    HumanMessagePromptTemplate.from_template(user_template)
]

qa_prompt = ChatPromptTemplate.from_messages(messages)

crc = ConversationalRetrievalChain.from_llm(
    llm=llm,  # Link the ChatGPT LLM
    retriever=retriever,  # Link the vector store based retriever
    memory=memory,  # Link the conversation memory
    chain_type='stuff',  # Specify the chain type
    combine_docs_chain_kwargs={'prompt': qa_prompt},
    verbose=True  # Set to True to enable verbose logging for debugging
)


In [34]:
db = load_embeddings_chroma()
q = 'How many pairs of questions and answer had the StackOverflow dataset? '
result = ask_question(q, crc)
print(result)



> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
System: 
Use the following pieces of context to answer the user's question.
---------------------
Context: ```million pairs of questions and answers. However, datasets do not
usually contain pre-existing question-and-answer or query-and-
candidate pairs in many real-world RAG scenarios. Therefore, it is vital

million pairs of questions and answers. However, datasets do not
usually contain pre-existing question-and-answer or query-and-
candidate pairs in many real-world RAG scenarios. Therefore, it is vital

simple similarity search was highly e ective because the dataset had 8
million pairs of questions and answers. However, datasets do not
usually contain pre-existing question-and-answer or query-and-

simple similarity search was highly e ective because the dataset had 8
million pairs of questions and answers. However, datasets do not
usually contain pre-existing question-and-an

In [35]:
q = 'When was Elon Musk born? '
result = ask_question(q, crc)
print(result)



> Entering new LLMChain chain...
Prompt after formatting:
Given the following conversation and a follow up question, rephrase the follow up question to be a standalone question, in its original language.

Chat History:

Human: How many pairs of questions and answer had the StackOverflow dataset? 
Assistant: The Stack Overflow dataset mentioned in the context had 8 million pairs of questions and answers.
Follow Up Input: When was Elon Musk born? 
Standalone question:

> Finished chain.


> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
System: 
Use the following pieces of context to answer the user's question.
---------------------
Context: ```February 13, 2024
Kaz Sato
Developer Advocate, Google Cloud
Guangsha Shi
Senior Product Manager, Google
Cloud
When a large language model (LLM) doesn’t have enough information
or has no contextual knowledge of a topic, it is more likely to hallucinate

February 13, 2024
Kaz Sato
Developer Ad